# Assignemnt 3 Part A - Business-Level Clustering

## A1 - Data Acquisition and Cleaning

### Data Exploration

In [1]:
import json
import pandas as pd

with open("data/business-licences.geojson") as f:
    data = json.load(f)

df = pd.json_normalize([f["properties"] for f in data["features"]])

In [2]:
df.shape


(204301, 27)

In [3]:
df.columns


Index(['folderyear', 'licencersn', 'licencenumber', 'licencerevisionnumber',
       'businessname', 'businesstradename', 'status', 'issueddate',
       'expireddate', 'businesstype', 'businesssubtype', 'unit', 'unittype',
       'house', 'street', 'city', 'province', 'country', 'postalcode',
       'localarea', 'numberofemployees', 'feepaid', 'extractdate', 'geom',
       'geo_point_2d', 'geo_point_2d.lon', 'geo_point_2d.lat'],
      dtype='object')

In [4]:
df.head()


,folderyear,licencersn,licencenumber,licencerevisionnumber,businessname,businesstradename,status,issueddate,expireddate,businesstype,...,country,postalcode,localarea,numberofemployees,feepaid,extractdate,geom,geo_point_2d,geo_point_2d.lon,geo_point_2d.lat
0,25,4637506,25-129785,00,(Anie Philip),None,Inactive,2024-11-25T21:02:01-08:00,2025-12-31,Long-term Rental,...,CA,V5R 3T5,Renfrew-Collingwood,0.0,94.0,2026-07-01T02:32:20-07:00,NaN,NaN,NaN,NaN
1,25,4637507,25-129786,00,(Tony Haughian),None,Issued,2024-11-18T09:52:13-08:00,2025-12-31,Long-term Rental,...,CA,V5T2N4,Mount Pleasant,0.0,226.0,2026-07-01T02:32:20-07:00,NaN,NaN,-123.088157,49.257736
2,25,4637508,25-129787,00,(Tony Haughian),None,Issued,2024-11-18T09:52:11-08:00,2025-12-31,Long-term Rental,...,CA,None,Mount Pleasant,0.0,275.0,2026-07-01T02:32:20-07:00,NaN,NaN,-123.089564,49.258053
3,25,4637511,25-129790,00,Simone Deborah Avram (Simone Avram),None,Issued,2025-01-13T15:28:19-08:00,2025-12-31,Long-term Rental,...,CA,V5V 2C6,Riley Park,0.0,188.0,2026-07-01T02:32:20-07:00,NaN,NaN,-123.096462,49.248896
4,25,4637516,25-129795,00,(William Squibb),None,Issued,2025-02-09T18:52:10-08:00,2025-12-31,Long-term Rental,...,CA,V5T 3Z3,Grandview-Woodland,0.0,437.0,2026-07-01T02:32:20-07:00,NaN,NaN,-123.066152,49.275461


In [5]:
df.dtypes

folderyear                object
licencersn                object
licencenumber             object
licencerevisionnumber     object
businessname              object
businesstradename         object
status                    object
issueddate                object
expireddate               object
businesstype              object
businesssubtype           object
unit                      object
unittype                  object
house                     object
street                    object
city                      object
province                  object
country                   object
postalcode                object
localarea                 object
numberofemployees        float64
feepaid                  float64
extractdate               object
geom                     float64
geo_point_2d             float64
geo_point_2d.lon         float64
geo_point_2d.lat         float64
dtype: object

In [6]:
df[["numberofemployees", "feepaid", "businesssubtype", "postalcode"]].isna().mean()


numberofemployees    0.000000
feepaid              0.369161
businesssubtype      0.894504
postalcode           0.466170
dtype: float64

Missingness varies a lot across these four columns. numberofemployees is complete. feepaid is missing for about 37% of records. businesssubtype is missing for about 89%, so it is too sparse to build a reliable type+subtype label from. postalcode is missing for about 47%, which limits how usable postal FSA would be as an area unit in Part B.

In [7]:
df["status"].value_counts(dropna=False)


status
Issued                  167179
Pending                  14791
Gone Out of Business     10535
Inactive                  6600
Cancelled                 5196
Name: count, dtype: int64

Issued accounts for 167,179 of 204,301 records, roughly 82%. The remaining statuses (Pending, Gone Out of Business, Inactive, Cancelled) do not represent currently valid licences, so they are candidates for exclusion.

In [8]:
df["businesstype"].value_counts()

businesstype
Long-term Rental                          45439
Health Care Professionals and Services    18766
General Contractor                        15908
Short-term Rental Operator                13419
Retail Dealer                              9458
                                          ...  
Urban Farm Class B                            8
Marine Service Station                        6
Adult Services                                4
Amusement Park                                3
Oil Gas and Other Fuels                       3
Name: count, Length: 94, dtype: int64

There are 94 business types with a very uneven distribution. A few categories dominate, led by Long-term Rental at over 45,000 records, while the smallest categories have fewer than ten. This long tail means the raw column needs consolidating before it can be used as a feature.

In [9]:
df["geo_point_2d.lat"].isna().mean()

np.float64(0.497535499092026)

About 50% of records have no latitude, and therefore no usable coordinates. Location clustering in A2 and the centroid map in B3 both require coordinates, and there is no sensible way to impute a location, so these rows cannot be used.

In [10]:
df["numberofemployees"].describe()


count    204301.000000
mean         10.021473
std          72.765566
min           0.000000
25%           0.000000
50%           1.000000
75%           4.000000
max        5876.000000
Name: numberofemployees, dtype: float64

The column is complete but heavily right skewed. The median is 1 employee while the mean is 10 and the maximum is 5,876. Most licences are very small operations with a small number of very large ones pulling the average up. This skew matters when scaling the feature later.

In [11]:
(df["numberofemployees"] == 0).mean()


np.float64(0.358294868845478)

About 36% of records report zero employees. Some of these are likely genuine, such as individual rental licences, but others may simply be unreported. The column still carries useful size information, though the zeros should be treated with some caution.

In [12]:
sub = df[df["geo_point_2d.lat"].notna() & (df["status"] == "Issued")]
print(sub.shape)
print(sub["feepaid"].isna().mean())

(85950, 27)
0.30810936591041305


Filtering to records that have coordinates and a status of Issued leaves 85,950 rows out of the original 204,301. Within that subset feepaid is still missing for about 31%, only slightly better than in the full data. Dropping those rows would cost another 26,000 records, so imputing the missing fees is the better trade.

In [13]:
sub["businesstype"].value_counts(normalize=True).cumsum().head(25)

businesstype
Health Care Professionals and Services           0.137510
Long-term Rental                                 0.256940
Retail Dealer                                    0.328400
Legal Services                                   0.389063
Restaurant                                       0.446294
Beauty Services                                  0.492158
Limited Service Food Establishment               0.536579
Business Support Services                        0.571111
Financial Services                               0.599058
Consulting and Management Services               0.626446
Real Estate Services                             0.650366
Retail Dealer - Food                             0.672903
General Contractor                               0.693729
Association or Society                           0.713775
Information Communication Technology             0.733636
Wholesale Dealer - Non-Food                      0.750599
Parking Area / Garage                            0.764491
H

The top 20 business types cover about 80% of the subset, and each category past that point adds less than 1%. Keeping the top 20 and grouping everything else as Other gives a manageable set of categories without losing much coverage.

The ranking has also shifted from the full dataset. Short-term Rental Operator was the fourth largest type overall but does not appear in the top 25 here, and General Contractor has dropped from third to thirteenth. The coordinate and status filters removed those categories more heavily than others, so the filtered data represents a somewhat different mix of industries than the raw file.

### Cleaning

In [14]:
df_clean = df[df["geo_point_2d.lat"].notna() & df["geo_point_2d.lon"].notna()].copy()


Rows without a latitude and longitude are dropped. A2 clusters businesses purely on location and B3 places each area at its centroid, so a record with no coordinates cannot be used in either. There is no reasonable way to guess a missing location. This removes about half the dataset, which is a large cut and worth keeping in mind when reading the results.

In [15]:
df_clean = df_clean[df_clean["status"] == "Issued"].copy()


Only licences with a status of Issued are kept. The other statuses cover licences that were never approved, were cancelled, or belong to businesses that have closed. Including them would mean clustering businesses that are not actually operating.

In [16]:
df_clean["feepaid_missing"] = df_clean["feepaid"].isna()
df_clean["feepaid"] = df_clean["feepaid"].fillna(df_clean["feepaid"].median())


Missing fees are filled with the median rather than dropped. About 31% of the remaining records have no fee recorded, and dropping them would cost another 26,000 rows on top of the coordinate loss. The median is used instead of the mean because the fee values are skewed. A flag column records which rows were imputed so the effect can be checked later.

In [17]:
top20 = df_clean["businesstype"].value_counts().head(20).index
df_clean["businesstype_grouped"] = df_clean["businesstype"].where(
    df_clean["businesstype"].isin(top20), "Other"
)

The top 20 business types are kept and everything else is grouped as Other. These 20 cover about 80% of the records, and every category beyond that adds less than 1% each. Leaving all 94 categories in place would create a very wide and mostly empty feature matrix once encoded. The top 20 are counted after filtering, not before, so the ranking reflects the data actually being used.

In [18]:
df_clean["issueddate"] = pd.to_datetime(
    df_clean["issueddate"], format="ISO8601", utc=True
).dt.tz_localize(None)
df_clean["expireddate"] = pd.to_datetime(df_clean["expireddate"], errors="coerce")


issueddate and expireddate are converted to datetime so that licence duration can be calculated in A3. issueddate includes a timezone offset that changes with daylight saving, so it is parsed as UTC and then stripped of the timezone. This is needed because expireddate has no timezone, and the two columns cannot be subtracted unless they match.

In [19]:
df_clean["licencerevisionnumber"] = pd.to_numeric(
    df_clean["licencerevisionnumber"], errors="coerce"
)

licencerevisionnumber is stored as text and is converted to a number so it can be used as a lifecycle feature.

In [20]:
print(df_clean.shape)
print(df_clean["businesstype_grouped"].value_counts())

(85950, 29)
businesstype_grouped
Other                                            17243
Health Care Professionals and Services           11819
Long-term Rental                                 10265
Retail Dealer                                     6142
Legal Services                                    5214
Restaurant                                        4919
Beauty Services                                   3942
Limited Service Food Establishment                3818
Business Support Services                         2968
Financial Services                                2402
Consulting and Management Services                2354
Real Estate Services                              2056
Retail Dealer - Food                              1937
General Contractor                                1790
Association or Society                            1723
Information Communication Technology              1707
Wholesale Dealer - Non-Food                       1458
Parking Area / Garage           

In [21]:
print(df_clean["expireddate"].isna().sum())
print(df_clean["issueddate"].isna().sum())
print((df_clean["expireddate"] - df_clean["issueddate"]).dt.days.describe())

148
147
count    85802.000000
mean       349.901273
std         81.582944
min       -354.000000
25%        344.000000
50%        377.000000
75%        399.000000
max        601.000000
dtype: float64


Note a negative min for duration. Data entry error. Let's check how many of these there are.

In [22]:
dur = (df_clean["expireddate"] - df_clean["issueddate"]).dt.days
print((dur < 0).sum())
print(dur[dur < 0].describe())

66
count     66.000000
mean     -35.772727
std       77.068161
min     -354.000000
25%      -22.000000
50%       -7.000000
75%       -2.250000
max       -1.000000
dtype: float64


66. Not too many. We'll drop these. 

In [23]:
df_clean["licence_duration"] = (df_clean["expireddate"] - df_clean["issueddate"]).dt.days
df_clean = df_clean[df_clean["licence_duration"] > 0].copy()
print(df_clean.shape)

(85689, 30)


There's the fully cleaned set. We'll save it now for user later. 

In [24]:
df_clean.to_parquet("data/businesses_clean.parquet")